In [68]:
import pandas as pd
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [142]:
df = pd.read_csv("baseline_keeper_dataset.csv")
df.sample(5)

,SHOT_XG_AT_PHASE_SECOND_BALL,name,BALL_WIN_REMOVED_OPPONENTS_DEFENDERS_AT_PHASE_SECOND_BALL,BALL_WIN_REMOVED_OPPONENTS_FROM_OPPONENT_PACKING_ZONE_FBR,current_team,DEF_PXT_PASS,REVERSE_PLAY_ADDED_OPPONENTS_FROM_PITCH_POSITION_MIDDLE_THIRD,BALL_WIN_REMOVED_OPPONENTS_IN_PITCH_POSITION_FINAL_THIRD,BYPASSED_DEFENDERS_RECEIVING_FROM_LANE_CENTER,BALL_LOSS_ADDED_OPPONENTS_DEFENDERS_FROM_PITCH_POSITION_MIDDLE_THIRD,...,SHOT_XG_BY_ACTION_MID_RANGE_SHOT,BYPASSED_OPPONENTS_BY_ACTION_THROW_IN,BALL_LOSS_ADDED_OPPONENTS_DEFENDERS_FROM_PITCH_POSITION_FIRST_THIRD,SHOT_XG_IN_LANE_RIGHT_HALF_SPACE,BYPASSED_OPPONENTS_RECEIVING_AT_PHASE_IN_POSSESSION,BYPASSED_OPPONENTS_RECEIVING_NUMBER_BY_ACTION_HEADER,BALL_LOSS_ADDED_OPPONENTS_BY_ACTION_SHORT_AERIAL_PASS,BALL_LOSS_REMOVED_TEAMMATES_DEFENDERS_FROM_PITCH_POSITION_OPPONENT_BOX,BYPASSED_DEFENDERS_RECEIVING_FROM_LANE_RIGHT_WING,origin_season
2867,NaN,Mio Backhaus,NaN,NaN,SV Werder Bremen,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-2024
1695,NaN,Jacob Karlstrøm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021
502,NaN,Remko Pasveer,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-2021
1319,NaN,Marco Carnesecchi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-2025
139,NaN,Lukas Hradecky,NaN,NaN,AS Monaco,NaN,NaN,NaN,NaN,NaN,...,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-2024


In [143]:
metadata_cols = [
    "playerId", "name", "birthdate", "status", "direction",
    "origin_team", "origin_comp", "origin_season", "current_team", 
    "current_comp", "current_season", "step", 
    "current_matches"
]
df['buy'] = ((df['direction'] == 'UP') | ((df['direction'] == 'NONE') & (df['step'] > 0.2))).astype(int)
X = df.drop(columns=metadata_cols + ["buy"])
y = df["buy"]
meta = df[metadata_cols]
print(df['buy'].value_counts(normalize=True) * 100)

buy
0    86.444182
1    13.555818
Name: proportion, dtype: float64


In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import numpy as np

# X, y already aggregated per player
# Assume df_agg is your aggregated dataframe
X = df.drop(columns=['playerId', 'step', 'buy', 'current_median', 'current_matches', 'origin_matches', 'origin_median', 'age'])
X = X.select_dtypes(include=[np.number])
y = df['buy']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Compute scale_pos_weight
n_0 = sum(y_train == 0)
n_1 = sum(y_train == 1)
scale_pos_weight = n_0 / n_1

# LightGBM dataset
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_test  = lgb.Dataset(X_test, label=y_test, reference=lgb_train)

# Train LightGBM classifier
params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbose': -1,
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42
}

model = lgb.train(
    params,
    lgb_train,
    num_boost_round=500,
    valid_sets=[lgb_train, lgb_test],
    valid_names=['train','test'],
)

# Predictions
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob >= 0.5).astype(int)

# Evaluation
print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_pred_prob))

# Feature importance
import pandas as pd
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importance(importance_type='gain')
}).sort_values(by='importance', ascending=False)

print("\nTop 20 KPIs driving buy decisions:")
print(importances.head(20))

              precision    recall  f1-score   support

           0       0.88      0.63      0.73       586
           1       0.17      0.48      0.25        92

    accuracy                           0.61       678
   macro avg       0.53      0.55      0.49       678
weighted avg       0.79      0.61      0.67       678

Confusion Matrix:
 [[368 218]
 [ 48  44]]
ROC-AUC Score: 0.6084266953553941

Top 20 KPIs driving buy decisions:
                                              feature    importance
53                                                age  10814.343995
38  BALL_LOSS_REMOVED_TEAMMATES_DEFENDERS_TO_OPPON...   1483.973934
40     BALL_LOSS_ADDED_OPPONENTS_AT_PHASE_SECOND_BALL   1045.900685
97  BYPASSED_OPPONENTS_RECEIVING_AT_PHASE_IN_POSSE...    972.127458
54  BALL_WIN_ADDED_TEAMMATES_DEFENDERS_BY_ACTION_H...    941.462162
64                BYPASSED_DEFENDERS_BY_ACTION_HEADER    880.562846
86                                   PXT_DRIBBLE_FAIL    862.495823
3                